<br>

# 1. Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings for better readability in notebooks
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.options.display.max_rows = 10000

<br>

# 2. Load Dataset

In [ ]:
# Load Stack Overflow Survey dataset
df = pd.read_csv('Data/stack-overflow-developer-survey-2020/survey_results_public.csv')

# Preview first rows
df

<br>

# 3. Data Preprocessing

<br>

###  &emsp; 3.1 Initial Data Exploration

In [ ]:
# View a random sample (transposed for better readability of all columns)
df.sample(1).T

In [ ]:
# Get dataset structure and data types
df.info()

####  Initial Observations

- The dataset contains a large number of features (~60+ columns)
- Many columns have `object` data types (categorical/text data)
- There are significant missing values across multiple columns

These observations indicate that data cleaning and feature selection are required.

<br>

### &emsp;  3.2 Selecting relevant columns for analysis

In [ ]:
# Select relevant columns for analysis
# Goal: Understand relationship between job roles (DevType) and technologies
# 'Respondent': unique id (tracking)
# 'Country': regional differences in tech usage


df_core = df[[
    'Respondent','Country','DatabaseDesireNextYear','DatabaseWorkedWith',
    'LanguageDesireNextYear','LanguageWorkedWith','MiscTechDesireNextYear',
    'MiscTechWorkedWith','PlatformDesireNextYear','PlatformWorkedWith',
    'WebframeDesireNextYear','WebframeWorkedWith','DevType'
]]
df_core

###  Feature Selection Strategy

Since the goal of this project is to understand the relationship between:
- Job roles
- Technologies (languages, tools, frameworks)

The dataset was reduced to include only features relevant to analyzing the relationship between job roles and technologies.<br>
<br>

#### Included:
- **DevType**: Target variable representing job roles
- **WorkedWith columns**: Technologies currently used by professionals
- **DesireNextYear columns**: Technologies professionals aim to learn<br>
<br>

#### Additional:
- **Respondent**: Retained as a unique identifier
- **Country**: Retained to allow potential regional analysis<br>
<br>

This structure enables both:
- Understanding current job requirements
- Exploring future skill trends

<br>

### &emsp;  3.3 Data Cleaning

#### &emsp;&emsp;3.3.1 Handle `DevType` Column (Target)


In [ ]:
# First, we inspect the unique values in the column to understand its structure.
df_core['DevType'].unique().tolist()

### Observations

- There are missing values (`NaN`) in the column
- The column contains multiple roles per entry (multi-label format)
- Most entries start with the word "Developer" followed by a specific role (e.g., "Developer, front-end; Developer, full-stack")
- Roles are separated by semicolons `;`<br>

<br>

### Plan

To clean this column, we will:
- Handle missing values
- Remove the prefix "Developer, " to simplify role names
- Split multiple roles into separate entries
- Standardize and structure the data to make it easier to analyze

In [ ]:
# Handle Missing Values in DevType

# Number of missing (null) and non-missing values
null_count = df_core['DevType'].isnull().sum()
not_null_count = df_core.shape[0] - null_count

print('Number of nulls:     ', null_count)
print('Number of not nulls: ', not_null_count)

#### Observations

- The `DevType` column contains a significant number of missing values (15,091 records)
- These missing values represent users without a defined job role
- Since `DevType` is the target variable, it is important to handle these values carefully before proceeding with the analysis
<br>

#### Plan

- Handle missing values in the `DevType` column by removing rows with null values, since `DevType` is the target variable and is required for analysis.

- Keep a separate dataset that contains the rows with missing `DevType` values, as these users may later be used for career recommendation.

In [ ]:
# Handle missing values in DevType

# separate rows with missing DevType (for future use)
df_no_devtype = df_core[df_core['DevType'].isnull()].copy()

# keep only rows with valid DevType (main dataset)
df_core = df_core[df_core['DevType'].notnull()].copy()
df_core

In [ ]:
# split DevType into multiple roles using ';' separator
df_core['DevType'] = df_core['DevType'].str.split(';')

In [ ]:
df_core

In [ ]:
# remove "Developer, " from each career
df_core['DevType'] = df_core['DevType'].apply(
    lambda careers: [career.replace('Developer, ', '').strip() for career in careers]
)

df_core

In [ ]:
# get frequency of each DevType role
df_core['DevType'].explode().value_counts()

In [ ]:
# count the number of careers (DevTypes) in each row
df_core['Career_nums'] = df_core['DevType'].apply(len)



In [ ]:
plt.hist(df_core['Career_nums'],bins=10)

#### Observations

- The distribution shows that most users have a small number of careers
- A small portion of users have a high number of careers, which may represent noisy or unrealistic data

#### Plan

- Use the 90th percentile as a threshold to identify outliers
- Remove rows where the number of careers exceeds this threshold (greater than 6)
- This helps reduce noise while keeping the majority of valid data

In [ ]:
# calculate the 90th percentile of Career_nums
threshold = df_core['Career_nums'].quantile(0.90)

print('90th percentile (threshold):', threshold)


# remove rows where number of careers exceeds the threshold
df_core = df_core[df_core['Career_nums'] <= threshold]

df_core